# 02 — Evaluation, Significance, and Calibration

In [ ]:
import sys, os, pickle
import numpy as np

candidates = [os.path.abspath("."), os.path.abspath("..")]
found = next((c for c in candidates
              if os.path.isfile(os.path.join(c, "cahgtn", "__init__.py"))), None)
if found is None:
    raise ModuleNotFoundError(
        f"Could not find a 'cahgtn' package folder in {candidates[0]} or "
        f"{candidates[1]}. See 00_Data_Pipeline.ipynb for the expected layout."
    )
sys.path.insert(0, found)
print(f"Using cahgtn package from: {os.path.join(found, 'cahgtn')}")

from cahgtn.evaluate import per_horizon_report, print_report

ALL_CITIES = ["Hangzhou", "Shanghai", "Beijing"]
available_cities = [c for c in ALL_CITIES if os.path.exists(f"{c}_predictions.pkl")]
missing_cities = [c for c in ALL_CITIES if c not in available_cities]
print(f"Found saved predictions for: {available_cities}")
if missing_cities:
    print(f"Not yet run (skipping): {missing_cities} -- run 00+01 for these when ready")
if len(available_cities) < 3:
    print("NOTE: a single-city result will not clear review at a strong venue --")
    print("      the paper's multi-city extension needs all three before submission.")

SOTA_REFERENCE = {
    "Hangzhou": {
        "PVCGN (per STR-GODEs' table)":      {"MAE": 32.68, "RMSE": 50.23, "MAPE_or_WMAPE": 9.67},
        "STR-GODEs (2021)":                   {"MAE": 31.68, "RMSE": 48.78, "MAPE_or_WMAPE": 9.31},
        "PVCGN (per Parallel-MGCN's table)":  {"MAE": 22.20, "RMSE": 38.12, "MAPE_or_WMAPE": 13.15},
        "PB-GRU (per Parallel-MGCN's table)": {"MAE": 22.13, "RMSE": 36.55, "MAPE_or_WMAPE": 13.30},
    },
    "Shanghai": {
        "PVCGN (per STR-GODEs' table)":      {"MAE": 23.41, "RMSE": 45.31, "MAPE_or_WMAPE": 17.04},
        "STR-GODEs (2021)":                   {"MAE": 23.21, "RMSE": 44.58, "MAPE_or_WMAPE": 16.99},
    },
}

In [ ]:
labels = ["15min", "30min", "45min", "60min"]

def aggregate(summary_for_name):
    horizon = len(summary_for_name[0])
    keys = summary_for_name[0][0].keys()
    agg = []
    for h in range(horizon):
        agg_h = {}
        for k in keys:
            vals = [run[h][k] for run in summary_for_name if not np.isnan(run[h][k])]
            if vals:
                agg_h[k] = (float(np.mean(vals)), float(np.std(vals)))
        agg.append(agg_h)
    return agg

def process_city(city):
    with open(f"{city}_predictions.pkl", "rb") as f:
        saved = pickle.load(f)
    all_predictions = saved["all_predictions"]
    naive_test = saved["naive_test"]
    head_type_for = saved["head_type_for"]

    summary = {}
    for name, runs in all_predictions.items():
        summary[name] = []
        for seed_idx, (preds, trues) in enumerate(runs):
            naive = naive_test if head_type_for.get(name, head_type_for.get(name.replace(" (extended)", ""))) == "nb" else None
            reports = per_horizon_report(preds, trues, naive_preds=naive)
            summary[name].append(reports)

    aggregated = {name: aggregate(summary[name]) for name in summary}

    print(f"\n{'Variant':<32} {'Horizon':>8} {'MAE':>14} {'RMSE':>14} {'MAPE%':>14} {'WMAPE%':>14}")
    for name, agg in aggregated.items():
        for h, rep in enumerate(agg):
            def fmt(k):
                return f"{rep[k][0]:.2f}+-{rep[k][1]:.2f}" if k in rep else "--"
            print(f"{name:<32} {labels[h]:>8} {fmt('MAE'):>14} {fmt('RMSE'):>14} {fmt('MAPE'):>14} {fmt('WMAPE'):>14}")

    return aggregated

per_city_results = {}
for city in available_cities:
    print(f"\n{'='*70}\nCITY: {city}\n{'='*70}")
    per_city_results[city] = process_city(city)

In [ ]:
for city in available_cities:
    aggregated = per_city_results[city]
    ref = SOTA_REFERENCE.get(city, {})
    print(f"\n=== 15-min horizon comparison, {city} ===")
    for name, agg in aggregated.items():
        rep = agg[0]
        def fmt(k):
            return f"{rep[k][0]:.2f}+-{rep[k][1]:.2f}" if k in rep else "--"
        print(f"{name:<32} MAE={fmt('MAE')}  RMSE={fmt('RMSE')}  MAPE={fmt('MAPE')}%  WMAPE={fmt('WMAPE')}%")
    if ref:
        for paper, m in ref.items():
            print(f"{paper:<32} MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}  MAPE/WMAPE={m['MAPE_or_WMAPE']:.2f}%")
    else:
        print(f"  (no literature reference numbers on file for {city})")

In [ ]:
from cahgtn.evaluate import ensemble_predictions

ensembled_by_city = {}
all_predictions_by_city = {}
naive_test_by_city = {}
for city in available_cities:
    with open(f"{city}_predictions.pkl", "rb") as f:
        saved = pickle.load(f)
    all_predictions_by_city[city] = saved["all_predictions"]
    naive_test_by_city[city] = saved["naive_test"]

    ensembled_by_city[city] = {}
    for name, runs in saved["all_predictions"].items():
        seed_preds = [p for p, _ in runs]
        trues = runs[0][1]
        ens = ensemble_predictions(seed_preds)
        ensembled_by_city[city][name] = (ens, trues)
print(f"Ensembled variants per city: {[(c, list(v.keys())) for c, v in ensembled_by_city.items()]}")

In [ ]:
from cahgtn.significance import compare_variants

COMPARISONS = [
    ("CA-HGTN (full)", "CA-HGTN (point head)"),    # does the NB head matter?
    ("CA-HGTN (full)", "CA-HGTN (single-graph)"),  # does multi-graph fusion matter?
    ("CA-HGTN (full)", "GRU-only baseline"),        # does spatial structure matter at all?
    ("CA-HGTN (full)", "GCN+GRU baseline"),         # beats a self-consistently-reproduced baseline?
]

for city in available_cities:
    all_predictions = all_predictions_by_city[city]
    print(f"\n=== {city}: significance tests (ensembled predictions, all 4 horizons) ===")
    for name_a, name_b in COMPARISONS:
        if name_a not in all_predictions or name_b not in all_predictions:
            print(f"  {name_a} vs {name_b}: skipped (one or both not trained for {city} yet)")
            continue
        preds_a = ensemble_predictions([p for p, _ in all_predictions[name_a]])
        preds_b = ensemble_predictions([p for p, _ in all_predictions[name_b]])
        trues = all_predictions[name_a][0][1]

        results = compare_variants(preds_a, preds_b, trues, loss="absolute")
        print(f"\n  {name_a}  vs  {name_b}:")
        for r in results:
            dm, wx = r["diebold_mariano"], r["wilcoxon"]
            direction = "A better" if dm["mean_loss_diff"] < 0 else "B better"
            print(f"    horizon {r['horizon_step']} ({r['horizon_step']*15}min): "
                  f"DM p={dm['p_value']:.4f} ({'SIGNIFICANT' if dm['significant_at_0.05'] else 'not significant'}, {direction})  "
                  f"Wilcoxon p={wx['p_value']:.4f}")

In [ ]:
from cahgtn.significance import bootstrap_relative_improvement

COMPARISON_PAIRS = [
    ("CA-HGTN (full)", "GCN+GRU baseline",
     "primary comparison: our own controlled reproduction"),
    ("GCN+GRU (bidirectional-attention)", "GCN+GRU baseline",
     "isolates temporal-component sophistication, both 5-seed results"),
    ("GCN+GRU (stacked)", "GCN+GRU (bidirectional-attention)",
     "does stacking add anything beyond a single bidirectional layer?"),
    ("CA-HGTN (full) (extended)", "CA-HGTN (full)",
     "does validation-searched extended training genuinely improve on the standard result -- "
     "check this alongside the calibration numbers (cov@90), not accuracy alone"),
]

for city in available_cities:
    for new_name, baseline_name, why in COMPARISON_PAIRS:
        if new_name not in ensembled_by_city[city] or baseline_name not in ensembled_by_city[city]:
            continue
        ens_new, trues_new = ensembled_by_city[city][new_name]
        ens_base, trues_base = ensembled_by_city[city][baseline_name]

        key_new = "mu" if "mu" in ens_new else "pred"
        key_base = "mu" if "mu" in ens_base else "pred"

        print(f"\n=== {city}: {new_name} improvement over {baseline_name} ({why}) ===")
        for h in range(trues_new.shape[1]):
            r = bootstrap_relative_improvement(
                ens_new[key_new][:, h], ens_base[key_base][:, h], trues_new[:, h], loss="absolute"
            )
            print(f"  {labels[h]}: {r['improvement_pct']:5.1f}% improvement, "
                  f"95% CI=[{r['ci_low']:5.1f}%, {r['ci_high']:5.1f}%], "
                  f"significant={r['significant']} (n={r['n_windows']} windows)")

In [ ]:
for city in available_cities:
    print(f"\n--- {city} ---")
    for name, agg in per_city_results[city].items():
        if "coverage@90" not in agg[0]:
            continue
        print(f"{name}:")
        for h, rep in enumerate(agg):
            print(f"  {labels[h]}: coverage@90={rep['coverage@90'][0]:.3f} (target 0.90)   "
                  f"coverage@50={rep['coverage@50'][0]:.3f} (target 0.50)   NB_NLL={rep['NB_NLL'][0]:.3f}")

In [ ]:
from cahgtn.evaluate import find_calibration_scale, apply_calibration_scale, nb_calibration, point_metrics

VARIANTS_TO_CALIBRATE = ["CA-HGTN (full)", "CA-HGTN (full) (extended)"]

for VARIANT_TO_CALIBRATE in VARIANTS_TO_CALIBRATE:
    for city in available_cities:
        if VARIANT_TO_CALIBRATE not in ensembled_by_city.get(city, {}):
            continue
        ens, trues = ensembled_by_city[city][VARIANT_TO_CALIBRATE]
        if "mu" not in ens:
            continue

        print(f"=== {city}: post-hoc calibration for {VARIANT_TO_CALIBRATE} ===")
        for h in range(trues.shape[1]):
            mu_h, r_h, true_h = ens["mu"][:, h], ens["r"][:, h], trues[:, h]
            n_windows = mu_h.shape[0]

            fit_idx = np.arange(n_windows) % 2 == 1
            report_idx = ~fit_idx

            c = find_calibration_scale(mu_h[fit_idx], r_h[fit_idx], true_h[fit_idx], target_coverage=0.90)

            before = nb_calibration(mu_h[report_idx], r_h[report_idx], true_h[report_idx], level=0.90)
            r_calibrated = apply_calibration_scale(r_h[report_idx], c)
            after = nb_calibration(mu_h[report_idx], r_calibrated, true_h[report_idx], level=0.90)
            mae_check = point_metrics(mu_h[report_idx], true_h[report_idx])["MAE"]
            width_growth_pct = 100 * (after["width@90"] / before["width@90"] - 1)

            print(f"  {labels[h]}: c={c:.3f}")
            print(f"      coverage@90: {before['coverage@90']:.3f} -> {after['coverage@90']:.3f}  (target 0.900)")
            print(f"      width@90:    {before['width@90']:.1f} -> {after['width@90']:.1f}  "
                  f"({width_growth_pct:+.0f}% -- check this isn't degenerate before trusting the coverage fix)")
            print(f"      MAE on this half: {mae_check:.2f} (unaffected by the correction by construction)")
        print()

In [ ]:
from cahgtn.evaluate import high_deviation_mask, apply_window_mask, per_horizon_report as phr

TOP_FRACTION = 0.2  # top 20% most-anomalous test windows

for city in available_cities:
    if "CA-HGTN (full)" not in ensembled_by_city[city]:
        continue
    ens_ref, trues_ref = ensembled_by_city[city]["CA-HGTN (full)"]
    mask = high_deviation_mask(trues_ref, naive_test_by_city[city], top_fraction=TOP_FRACTION)
    print(f"\n=== {city}: {mask.sum()}/{len(mask)} windows ({mask.mean()*100:.0f}%) flagged high-deviation ===")

    for name in ["CA-HGTN (full)", "GCN+GRU baseline"]:
        if name not in ensembled_by_city[city]:
            continue
        ens, trues = ensembled_by_city[city][name]
        if "mu" in ens:
            d_routine = {"mu": ens["mu"][~mask], "r": ens["r"][~mask]}
            d_anom    = {"mu": ens["mu"][mask],  "r": ens["r"][mask]}
        else:
            d_routine = {"pred": ens["pred"][~mask]}
            d_anom    = {"pred": ens["pred"][mask]}
        routine = phr(d_routine, trues[~mask])
        anomalous = phr(d_anom, trues[mask])
        print(f"  {name:<28} routine MAE={routine[0]['MAE']:.2f}   high-deviation MAE={anomalous[0]['MAE']:.2f}")

## Single-station forecast plot

In [ ]:
from cahgtn.evaluate import plot_station_forecast

CITY_TO_PLOT = available_cities[0] if available_cities else None
VARIANT_TO_PLOT = "CA-HGTN (full)" #varient type
STATION_IDX = 12   # station index
HORIZON_IDX = 0    # 0=15min, 1=30min, 2=45min, 3=60min
FLOW_IDX = 0       # 0=inflow, 1=outflow

if CITY_TO_PLOT and VARIANT_TO_PLOT in ensembled_by_city.get(CITY_TO_PLOT, {}):
    ens, trues = ensembled_by_city[CITY_TO_PLOT][VARIANT_TO_PLOT]
    fig = plot_station_forecast(
        ens, trues, station_idx=STATION_IDX, horizon_idx=HORIZON_IDX, flow_idx=FLOW_IDX,
        save_path=f"station_{STATION_IDX}_{VARIANT_TO_PLOT.replace(chr(32), chr(95))}.png",
    )
else:
    print(f"'{VARIANT_TO_PLOT}' not available yet for plotting.")